In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(bool(os.getenv("ANTHROPIC_API_KEY")))

In [4]:
import os

print(bool(os.getenv("ANTHROPIC_API_KEY")))

True


In [5]:
# Create an API clientassistant
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [6]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature= None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 2000,
        "messages": messages,
        "stop_sequences" : stop_sequences
    }
    
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""  # fallback if no text block found

In [7]:
messages = []

prompt = """
Generate three different samples AWS CLI commands. 
Each should be very short. 
Remove the backticks, bash, and newlines.
Give me the commands by themselves
"""  

add_user_message(messages, prompt)

text = chat(messages)
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws iam list-users'

In [8]:
from IPython.display import Markdown

Markdown(text)

aws s3 ls
aws ec2 describe-instances
aws iam list-users

In [9]:
messages = []
add_user_message(messages, "write a one sentence description of a fake database")

stream=client.messages.create(
    model=model,
    max_tokens=1000, 
    messages=messages,
    stream=True 
)
for event in stream: 
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CePrmKW9pzVSy31vTToDQ', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=3, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fict', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ional database called "EchoVault" secretly stores every deleted text message ever sent, organized by regret level.', type='text_delta'), index=0, type='content_block_delta')
RawCont

In [10]:
messages = []
add_user_message(messages, "write a one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000, 
    messages=messages
) as stream:
    for text in stream.text_stream: 
        # print(text, end="")
        pass

    final_message = stream.get_final_message()

for block in final_message.content:
    if block.type == "text":
        print(block.text)


A sprawling NoSQL database called "WhisperDB" that claims to store and instantly retrieve memories, secrets, and half-forgotten dreams by indexing them according to emotional resonance rather than timestamps or keys.


In [11]:
messages = []

add_user_message(messages, 
"Generate a very short event bridge rule as json"
"Respond with only the JSON object, no explanation, no markdown code fences.")
chat(messages)

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [12]:
result = chat(messages)
result = result.strip().removeprefix("```json").removesuffix("```").strip()
print(result)

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}


In [13]:
messages = []
add_user_message(
    messages,
    "Generate a two line joke.",
)
answer = chat(messages)

answer

"**Setup:** I told my computer I needed a break.\n\n**Punchline:** Now it won't stop sending me vacation ads."

In [14]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
answer = chat(messages)

answer


'Unlike classical bits, which represent either a 0 or a 1, quantum bits (qubits) can exist in multiple states simultaneously, enabling quantum computers to explore many possible solutions at once.'

In [15]:
import json


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output:

[
    {
            "task": "Description of task",
            "format": "python",
            "solution_criteria": "Description of what a good solution must contain"

        
    }
]

* Include a "format" field for every object.
* The format must be exactly one of: "python", "json", or "regex".
* Include a "solution_criteria" field for every object describing what a strong solution must contain.
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code.

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    text = text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(text)


In [16]:
dataset=generate_dataset()

with open("dataset.json", "w") as f: 
          json.dump(dataset, f, indent=2)

In [17]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

The required response format is: {test_case["format"]}

Respond only with the requested {test_case["format"]} content.
Do not include markdown code fences.
Do not add comments, commentary, explanations, headings, or introductory text.
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [18]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Solution Criteria:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with valid JSON only. Do not use markdown code fences.

    Example response shape:
    {{
        "strengths": ["example strength"],
        "weaknesses": ["example weakness"],
        "reasoning": "concise explanation",
        "score": 8
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages)

    return json.loads(eval_text)


In [19]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format_type = test_case["format"]

    if format_type == "json":
        return validate_json(response)
    elif format_type == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [20]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }

In [21]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        average_score = mean(result["score"] for result in results)
        print(f"Average score: {average_score}")
        
    return results

In [22]:
results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 6.0
Average score: 7.75
Average score: 8.5
[
  {
    "output": "def is_log_file_under_logs_prefix(key):\n    return key.startswith('logs/') and key.endswith('.log') and len(key) > len('logs/.log') - 1 and key[len('logs/'):] != ''",
    "test_case": {
      "task": "Write a Python function that takes an S3 object key (string) and returns True if it represents a file with a '.log' extension located under a 'logs/' prefix, and False otherwise. For example, 'logs/2023/app.log' should return True, but 'data/app.log' or 'logs/app.txt' should return False.",
      "format": "python",
      "solution_criteria": "Defines a function accepting a string parameter; correctly checks that the key starts with 'logs/' and ends with '.log'; returns a boolean True/False (not truthy/falsy values); handles edge cases like keys that are exactly 'logs/' or lack an extension without raising errors."
    },
    "model_score": 2,
    "syntax_score": 10,
    "score": 6.0,
    "reasoning": "The cor